# Carteira de Investimentos — Perfil Conservador

Análise quantitativa de uma carteira hipotética de **R$ 100.000**, montada com perfil de investidor conservador, cobrindo renda fixa (Tesouro Selic, CDB, LCI) e renda variável (BBSE3, GGBR4).

Este notebook cobre:
- Rentabilidade líquida de cada ativo no período
- Retorno consolidado da carteira
- Volatilidade (desvio-padrão) dos ativos de renda variável
- Correlação de Pearson entre os ativos de renda variável
- Comparação de eficiência entre produtos de renda fixa (LCI isenta vs. CDB tributado)

> Projeto acadêmico (disciplina de Investimento e Risco) adaptado para portfólio técnico — o texto dissertativo original (cenário macro, justificativas) foi removido; aqui ficam apenas os cálculos.

## 1. Parâmetros do período

CDI e IPCA mensais utilizados no cálculo dos rendimentos de renda fixa.

In [1]:
import math

# CDI mensal (abril e maio)
cdi_abril = 0.0109
cdi_maio  = 0.0092
cdi_periodo = (1 + cdi_abril) * (1 + cdi_maio) - 1

# IPCA mensal (abril e maio) — usado como referência de inflação do período
ipca_abril = 0.0088
ipca_maio  = 0.0067
ipca_periodo = (1 + ipca_abril) * (1 + ipca_maio) - 1

# Alíquota de IR para o prazo da aplicação
ir = 0.225

print(f"CDI acumulado no período: {cdi_periodo*100:.2f}%")
print(f"IPCA acumulado no período: {ipca_periodo*100:.2f}%")

CDI acumulado no período: 2.02%
IPCA acumulado no período: 1.56%


## 2. Composição da carteira e rendimento da renda fixa

Alocação original de R$ 100.000 entre os seis ativos.

In [2]:
# --- Tesouro Selic: 35% da carteira, 100% do CDI ---
ts_valor = 35000
ts_taxa = cdi_periodo
ts_ganho = ts_valor * ts_taxa * (1 - ir)
ts_vf = ts_valor + ts_ganho

# --- CDB: 25% da carteira, 110% do CDI ---
cdb_valor = 25000
cdb_taxa = (1 + cdi_abril * 1.10) * (1 + cdi_maio * 1.10) - 1
cdb_ganho = cdb_valor * cdb_taxa * (1 - ir)
cdb_vf = cdb_valor + cdb_ganho

# --- Renda fixa pós-fixada adicional: 10% da carteira, 120% do CDI ---
rf3_valor = 10000
rf3_taxa = (1 + cdi_abril * 1.20) * (1 + cdi_maio * 1.20) - 1
rf3_ganho = rf3_valor * rf3_taxa * (1 - ir)
rf3_vf = rf3_valor + rf3_ganho

# --- LCI: 10% da carteira, 95% do CDI, isenta de IR ---
lci_valor = 10000
lci_taxa = (1 + cdi_abril * 0.95) * (1 + cdi_maio * 0.95) - 1
lci_ganho = lci_valor * lci_taxa   # sem desconto de IR
lci_vf = lci_valor + lci_ganho

for nome, ganho, vf in [
    ("Tesouro Selic", ts_ganho, ts_vf),
    ("CDB 110% CDI", cdb_ganho, cdb_vf),
    ("Renda fixa 120% CDI", rf3_ganho, rf3_vf),
    ("LCI 95% CDI (isenta)", lci_ganho, lci_vf),
]:
    print(f"{nome:22s} | ganho líquido: R$ {ganho:8.2f} | valor final: R$ {vf:10.2f}")

Tesouro Selic          | ganho líquido: R$   547.93 | valor final: R$   35547.93
CDB 110% CDI           | ganho líquido: R$   430.73 | valor final: R$   25430.73
Renda fixa 120% CDI    | ganho líquido: R$   188.05 | valor final: R$   10188.05
LCI 95% CDI (isenta)   | ganho líquido: R$   191.86 | valor final: R$   10191.86


## 3. Renda variável — BBSE3 e GGBR4

Cotações semanais no período analisado (8 pontos por ativo) e retorno com base em entrada/saída de R$ 10.000 em cada.

In [3]:
# Cotações semanais de fechamento
bbse3_precos = [34.00, 33.75, 33.90, 34.30, 34.50, 34.80, 34.90, 35.23]
ggbr4_precos = [19.50, 20.00, 21.00, 21.96, 23.50, 24.36, 24.00, 23.75]

bbse3_valor = 10000
ggbr4_valor = 10000

# Quantidade de cotas/ações compradas no preço de entrada
bbse3_qtd = bbse3_valor / bbse3_precos[0]
ggbr4_qtd = ggbr4_valor / ggbr4_precos[0]

bbse3_retorno = (bbse3_precos[-1] / bbse3_precos[0]) - 1
ggbr4_retorno = (ggbr4_precos[-1] / ggbr4_precos[0]) - 1

# Dividendos pendentes de GGBR4 (anunciados no período, pagamento futuro)
ggbr4_divs = 227.16  # valor de proventos pendentes anunciados

print(f"BBSE3: {bbse3_precos[0]:.2f} -> {bbse3_precos[-1]:.2f} | retorno: {bbse3_retorno*100:.2f}%")
print(f"GGBR4: {ggbr4_precos[0]:.2f} -> {ggbr4_precos[-1]:.2f} | retorno: {ggbr4_retorno*100:.2f}%")
print(f"GGBR4 dividendos pendentes: R$ {ggbr4_divs:.2f}")
print(f"Retorno GGBR4 com dividendos pendentes: {(ggbr4_retorno*ggbr4_valor + ggbr4_divs)/ggbr4_valor*100:.2f}%")

BBSE3: 34.00 -> 35.23 | retorno: 3.62%
GGBR4: 19.50 -> 23.75 | retorno: 21.79%
GGBR4 dividendos pendentes: R$ 227.16
Retorno GGBR4 com dividendos pendentes: 24.07%


## 4. Volatilidade e correlação entre os ativos de renda variável

Desvio-padrão dos retornos semanais (anualizado por raiz de 52 semanas) e correlação de Pearson entre BBSE3 e GGBR4.

In [4]:
def retornos_semanais(precos):
    return [(precos[i+1] - precos[i]) / precos[i] for i in range(len(precos)-1)]

def media(lst):
    return sum(lst) / len(lst)

def desvio_padrao(lst):
    mu = media(lst)
    return (sum((x - mu)**2 for x in lst) / (len(lst) - 1)) ** 0.5

bbse3_r = retornos_semanais(bbse3_precos)
ggbr4_r = retornos_semanais(ggbr4_precos)

m1, s1 = media(bbse3_r), desvio_padrao(bbse3_r)
m2, s2 = media(ggbr4_r), desvio_padrao(ggbr4_r)

covariancia = sum((bbse3_r[i]-m1)*(ggbr4_r[i]-m2) for i in range(len(bbse3_r))) / (len(bbse3_r)-1)
correlacao = covariancia / (s1 * s2)

print(f"BBSE3 | retorno médio semanal: {m1*100:.3f}% | desvio-padrão semanal: {s1*100:.3f}% | anualizado: {s1*math.sqrt(52)*100:.1f}%")
print(f"GGBR4 | retorno médio semanal: {m2*100:.3f}% | desvio-padrão semanal: {s2*100:.3f}% | anualizado: {s2*math.sqrt(52)*100:.1f}%")
print(f"Covariância: {covariancia:.7f}")
print(f"Correlação de Pearson (ρ): {correlacao:.4f}")

BBSE3 | retorno médio semanal: 0.511% | desvio-padrão semanal: 0.630% | anualizado: 4.5%
GGBR4 | retorno médio semanal: 2.898% | desvio-padrão semanal: 3.149% | anualizado: 22.7%
Covariância: 0.0000205
Correlação de Pearson (ρ): 0.1036


## 5. Eficiência de renda fixa: LCI isenta vs. CDB tributado

Comparação entre um produto isento de IR (LCI) e um equivalente tributável (CDB), para verificar qual entrega mais retorno líquido.

In [5]:
equiv_lci_cdb = lci_taxa / cdi_periodo  # % do CDI que um CDB precisaria pagar (bruto) para igualar a LCI líquida
cdb_100_liquido = cdi_periodo * (1 - ir)

print(f"LCI 95% CDI (líquida): {lci_taxa*100:.2f}%")
print(f"Equivale a um CDB de aproximadamente {equiv_lci_cdb*100:.1f}% do CDI (bruto) para entregar o mesmo líquido")
print(f"CDB 100% CDI líquido de IR: {cdb_100_liquido*100:.2f}%")
print(f"LCI supera o CDB 100% CDI líquido: {lci_taxa > cdb_100_liquido}")

LCI 95% CDI (líquida): 1.92%
Equivale a um CDB de aproximadamente 95.0% do CDI (bruto) para entregar o mesmo líquido
CDB 100% CDI líquido de IR: 1.57%
LCI supera o CDB 100% CDI líquido: True


## 6. Retorno consolidado da carteira

In [6]:
valor_inicial = 100000
ganho_total = (
    ts_ganho + cdb_ganho + rf3_ganho + lci_ganho
    + bbse3_retorno * bbse3_valor
    + ggbr4_retorno * ggbr4_valor
)
retorno_total = ganho_total / valor_inicial

ganho_total_com_divs = ganho_total + ggbr4_divs
retorno_total_com_divs = ganho_total_com_divs / valor_inicial

print(f"Ganho total no período: R$ {ganho_total:.2f}")
print(f"Retorno total da carteira: {retorno_total*100:.2f}%")
print(f"Retorno total com dividendos pendentes de GGBR4: {retorno_total_com_divs*100:.2f}%")

Ganho total no período: R$ 3899.82
Retorno total da carteira: 3.90%
Retorno total com dividendos pendentes de GGBR4: 4.13%


## Notas

- Os valores de CDI e IPCA correspondem aos meses de abril e maio de referência do período analisado.
- O IR (22,5%) foi aplicado conforme a tabela regressiva para aplicações de curto prazo; ativos isentos (LCI) não sofrem esse desconto.
- Os preços de BBSE3 e GGBR4 são cotações semanais de fechamento no período de acompanhamento.
- Este notebook é a versão "código" do projeto acadêmico; o texto de contexto (cenário macroeconômico, justificativa de ativos) foi mantido apenas no documento Word entregue à disciplina, não neste repositório.